# 03 — Recoding: from raw text to usable numbers

**Purpose.** Show what `daa.features.recode` does, why it is allowed to run
outside the cross-validation folds, and what the training frame looks like
after it.

**Read `02_raw_eda.ipynb` first.** That notebook describes the raw files. This
one changes them.

## Why recoding exists

The survey export stores numbers as text. `BM2` (weight) is a string column
with 102 levels, and `'70.0'` is one of those levels. A model that one-hot
encodes it learns "the level 70.0", not "70 kilograms".

Some cells are not answers:

- `'Sem resposta'` means "no answer".
- `'Não aplicável'` means "this question did not apply to this person".

These are *routing* facts. The survey skips questions, and who was skipped is
often related to age. So the fact is useful and must not be erased. We keep it
in a companion `*_FLAG` column.

## The design principle

> A transform leaks when it **estimates** something from rows it must not see
> (a median, a category frequency). Recoding is a fixed rule-book: the same
> input value always produces the same output value, no matter which rows are
> in the frame. Nothing is estimated, so nothing can leak. Therefore recoding
> may run at load time, outside the CV folds. Anything that **learns from
> data** (imputation, rare-level grouping, scaling, target encoding) stays in
> `preprocess.py` inside the pipeline, where it is refitted on each fold.

## Rules of this notebook

- Use `load_train()` and `load_test()`. They now return the recoded frame.
- The one exception is the before/after comparison: it reads a few raw columns
  with `pd.read_csv(..., usecols=[...])`, clearly marked. That raw frame is
  only for display. Never pass it to a model.
- Nothing here is written to disk.

## 0. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from daa.config import TRAIN_PATH
from daa.data import explicit_spec, infer_column_spec, load_test, load_train
from daa.features.recode import (
    BUCKET_COLUMNS,
    HHMM_COLUMNS,
    PARSED_COLUMNS,
    QUANTITY_COLUMNS,
    SENTINELS,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

if not TRAIN_PATH.exists():
    raise SystemExit("data/ is missing; this notebook needs the real files")

X, y = load_train()
X_test, row_ids = load_test()

print(f"train: {X.shape[0]:,} rows x {X.shape[1]} columns")
print(f"test:  {X_test.shape[0]:,} rows x {X_test.shape[1]} columns")
print(f"target: {y.nunique()} classes")
print(f"sentinels kept as flag levels: {SENTINELS}")

## 1. The three column groups

Every rule is a module-level constant, so the whole rule-book fits on one
screen. Columns outside these groups are returned unchanged.

| group | rule |
|---|---|
| `HHMM_COLUMNS` | fixed-width `'HHMM'` text read as a **duration** in minutes: `60*HH + MM` |
| `QUANTITY_COLUMNS` | plain numbers stored as text |
| `BUCKET_COLUMNS` | numbers plus a `'>=N'` / `'N ou mais'` bucket, which becomes its floor `N` |

Each parsed column gets a `<COL>_FLAG` companion with the levels `answer`,
`Sem resposta`, `Não aplicável`, and `missing` (a real gap in the file).

In [ ]:
groups = pd.DataFrame(
    {
        "HHMM (duration)": pd.Series(HHMM_COLUMNS),
        "quantity": pd.Series(QUANTITY_COLUMNS),
        "bucket (floor)": pd.Series(BUCKET_COLUMNS),
    }
)
print(f"{len(PARSED_COLUMNS)} parsed columns, each with a _FLAG companion")
display(groups)

## 2. Before and after: `BM2` (weight)

`BM2` was a 102-level string column. After recoding it is a float column plus
a flag column.

The flag matters: a person whose weight is missing because the question was
skipped is not the same as a person who refused to answer.

> The next cell reads the raw CSV on purpose, only to show the "before" side.
> The frame it builds is never used as model input.

In [ ]:
# Read-only: raw values, for the "before" column only.
raw_bm2 = pd.read_csv(TRAIN_PATH, usecols=["BM2"], low_memory=False)["BM2"]

before_after = pd.DataFrame(
    {
        "before": [
            f"{raw_bm2.nunique(dropna=False)} string levels",
            f"{raw_bm2.isna().sum()} real NaN",
            "not numeric: one-hot would create 102 dummy columns",
        ],
        "after": [
            f"{X['BM2'].nunique()} distinct float values",
            f"{X['BM2'].isna().sum()} NaN (sentinels became NaN)",
            "numeric: one column, usable by any model",
        ],
    },
    index=["levels", "missing", "model view"],
)
display(before_after)
display(X[["BM2", "BM2_FLAG"]].head(8))
display(X["BM2_FLAG"].value_counts().to_frame("rows"))

## 3. `PE7` / `PE9`: durations, not clock times

`PE7` and `PE9` are written as `'HHMM'`. We read them as **hours and
minutes**, that is `60*HH + MM` minutes.

Why a duration and not a clock time? Because values above 23:59 exist:
`'2400'`, `'2425'`, `'2500'`, `'3000'`. Those are impossible on a clock, but
plausible as totals. The data also shows half-hour codes (`'0130'`) while
raw-minute codes (`'0060'`) never appear.

Consequence: no cyclical sin/cos encoding is ever needed for these columns.
The linear minutes are final. Values above 23:59 are real answers and are
kept, not clipped.

Still open: *which* activity the duration measures. The parse is safe; the
report interpretation needs the survey documentation.

In [ ]:
# Read-only: raw values, to list the rows above 23:59 before recoding.
raw_pe = pd.read_csv(TRAIN_PATH, usecols=list(HHMM_COLUMNS), low_memory=False)

above_2359 = {}
for column in HHMM_COLUMNS:
    text = raw_pe[column][raw_pe[column].str.fullmatch(r"\d{4}")]
    above_2359[column] = text[text.str[:2].astype(int) > 23].value_counts()

print("raw values above 23:59 (kept as real answers):")
display(pd.DataFrame(above_2359).fillna(0).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for axis, column in zip(axes, HHMM_COLUMNS):
    axis.hist(X[column].dropna(), bins=60, color="#4477aa")
    axis.set_title(f"{column} — minutes (60*HH + MM)")
    axis.set_xlabel("minutes")
    axis.set_ylabel("rows")
plt.tight_layout()
plt.show()

display(X[list(HHMM_COLUMNS)].describe())

## 4. Buckets: `'>=7'` becomes `7.0`

`HHNBPERS_COD` (household size) mixes numbers with a `'>=7'` bucket. The exact
value is unknown, so we use the only number we can defend: the floor, 7. This
is an approximation, not a true value.

`IN25` and `IN26` use the Portuguese spelling `'53 ou mais'` / `'33 ou mais'`,
and `HO12_COD` / `HO34_COD` use `'>= 25'` / `'>= 60'` (with a space). The same
rule covers all of them.

In [ ]:
# Read-only: raw values, to show each level next to its recoded value.
raw_hh = pd.read_csv(TRAIN_PATH, usecols=["HHNBPERS_COD"], low_memory=False)["HHNBPERS_COD"]
levels = raw_hh.value_counts()

display(
    pd.DataFrame(
        {
            "raw level": levels.index,
            "raw rows": levels.values,
            "recoded value": [X["HHNBPERS_COD"][raw_hh == level].iloc[0] for level in levels.index],
        }
    )
)

bucket_rows = []
for column in BUCKET_COLUMNS:
    raw = pd.read_csv(TRAIN_PATH, usecols=[column], low_memory=False)[column]
    text = raw[raw.str.contains(r">=|ou mais", regex=True)]
    bucket_rows.append(
        {
            "column": column,
            "bucket levels": " | ".join(sorted(text.unique())),
            "rows in bucket": len(text),
            "recoded to": sorted(X[column][text.index].unique().tolist()),
        }
    )
display(pd.DataFrame(bucket_rows))

## 5. `explicit_spec`: the hand-written overrides

`infer_column_spec` only looks at dtypes and constants. It cannot know that a
column is admin metadata. `explicit_spec` adds those calls by hand, each with
its reason:

- `REFYEAR`, `REFMONTH`: interview year and month. Admin metadata, not a
  property of the person.
- `IN13`: 99.94% one value (7 informative rows out of 10,962).
- `AC1A`: 99.03% one value. Revisit only with a domain justification.

`run_experiment` now uses `explicit_spec(infer_column_spec(X))`.

The `ordinal` group stays empty on purpose: a level order is a domain claim,
so someone must confirm it from the survey documentation first.

In [ ]:
inferred = infer_column_spec(X)
explicit = explicit_spec(inferred)

roles = pd.DataFrame(
    {"inferred": pd.Series(inferred.counts()), "explicit": pd.Series(explicit.counts())}
)
display(roles)

added_drops = sorted(set(explicit.drop) - set(inferred.drop))
print(f"columns dropped by explicit_spec: {added_drops}")
print(f"input columns: {len(inferred.input_columns)} -> {len(explicit.input_columns)}")

## 6. Optional: one quick end-to-end run

Five fits on repeat 0 only: the point is to see the whole pipeline work on the
recoded frame, not to tune anything.

`preprocessor="passthrough"` keeps numeric NaN in place, which
`HistGradientBoostingClassifier` handles natively.

`save=False` avoids writing a run directory and a leaderboard row.

This takes a few minutes. Set `RUN_QUICK = False` to skip it.

In [ ]:
RUN_QUICK = True

if RUN_QUICK:
    from daa.experiment import run_experiment
    from daa.splits import get_folds

    folds = [fold for fold in get_folds(y) if fold.repeat == 0]
    quick = run_experiment(
        "hist_gb",
        name="hist_gb_repeat0_recoded",
        preprocessor="passthrough",
        X=X,
        y=y,
        folds=folds,
        save=False,
    )
    display(pd.Series(quick.metrics, name="hist_gb, repeat 0"))

## 7. Caveats

1. The recoding is a fixed rule-book, so it does not leak. The preprocessor
   (imputation, scaling, one-hot) still runs inside each fold.
2. Sentinels and routing are legitimate signal: they exist in train and in
   test.
3. The `AGE_RANGE` band-0 dispute (see `TODO.md`) does not block anything
   here: labels are used as codes.
4. Old `data_v1` numbers in `archive/` are **accuracy**. All new comparisons
   use macro F1.

## 8. Codebook flags

The data tells us the *shape*, not the *meaning*. These fields are parsed or
kept, but the interpretation is provisional until the survey codebook confirms
it. Do not state any of these as fact in the report.

| column / group | what the data shows | what the codebook must confirm | risk if wrong |
|---|---|---|---|
| `PE7`, `PE9` | fixed-width `HHMM`; values above 23:59; half-hour codes | which activity the duration measures; whether it is a total | the duration reading is confirmed, the report interpretation is not |
| `BM1`, `BM2` | height/weight-like values | units (cm/kg) and whether self-reported | derived features (BMI) would be wrong |
| `IN55`, `IN56`, `IN57`, `IN45`, `IN4`, `IN3`, `IN59`, `IN2`, `IN47` | numeric-looking, many levels | units and meaning; counts, amounts, or codes | a code treated as a quantity imposes a false order |
| `IN29` | 4 numeric levels | rating scale or count | ordinal vs numeric choice |
| `AM3`, `AM5`, `DH2`, `DH4`, `PE2`, `PE4`, `PE6`, `PE8`, `SK2` | integer levels | rating scales (ordinal) or counts | ordinal promotion later |
| `HHNBPERS_COD`, `HHNBPERS_0_13_COD` | small integers with `>=` buckets; names suggest household size / children | whether the bucket floor is acceptable | the floor is an approximation, not a true value |
| `IN25`, `IN26` | numeric values with `'N ou mais'` buckets | what they count; whether the floor is acceptable | same |
| `HO12_COD`, `HO34_COD` | `_COD` columns with `>= 25` / `>= 60` buckets | counts or nominal codes | parsing as numeric may impose a false order |
| `PA1` | `YYYYMM` + text buckets (`'Há mais de 1 ano'`, `'Nunca'`) | what the date refers to; the bucket boundaries | a future "months since" feature would be wrong |
| `IN24` | years 1997–2008 | what the year refers to (birth? event?) | a future "years since" feature would be wrong |
| `IN21`, `IN62` | multi-select categories | whether multiple answers are allowed; the full list | splitting into binary indicators would be wrong |
| `IN13` | 99.94% `'Não aplicável'`, 7 informative rows | what the 7 rows mean | dropping could lose a rare signal; keeping could overfit |
| `AC1A` | 99.03% `'Não'` | what it measures | dropping could lose a rare signal |
| `IN16` | 51.1% real NaN | whether the NaN is a visit/no-visit pattern | the missingness is implicit in the one-hot encoder |
| `HS1`-style rating scales | ordered-looking levels | the level order | ordinal promotion must wait for the codebook |
| `SENTINELS` | `'Sem resposta'`, `'Não aplicável'`, `'NR'`, the long `'Não aplicável (...)'` variant | which are routing, which are non-response, which are substantive skips | treating a substantive skip as routing (or vice versa) changes the flag meaning |
| `AGE_RANGE` | 16 bands, band 0 disputed | the official band mapping | already tracked in `TODO.md`; not a recode decision |

## 9. How this may change when we optimise for the best score

Mirrors the module docstring of `daa/features/recode.py`:

1. `PE7` / `PE9`: the duration reading is confirmed by the data, so no
   cyclical sin/cos encoding is needed and the linear minutes are final. Still
   open: which activity the duration measures. Verify against the survey
   documentation before interpreting results in the report.
2. If the 29 flag columns bloat linear models, collapse them to a single
   binary "answered" flag. Keep the full flags for trees.
3. Parse the left-nominal columns once their meaning is confirmed: `PA1`
   (`YYYYMM` plus buckets), `IN24` (years), `IN21` / `IN62` (split into binary
   indicators). `HO12_COD` / `HO34_COD` are already parsed as buckets in v1;
   any rare-code grouping is a learned transform and belongs in
   `preprocess.py`.
4. Ordinal promotion of rating scales (`HS1`-style) lives in `explicit_spec`,
   never in `recode.py`. The long `'Não aplicável (...)'` variant in `HA1A`–`HA1G`
   must be handled before those columns are promoted.
5. Anything that learns from the data (rare-level grouping, target encoding)
   must never be added to `recode.py`. It belongs in `preprocess.py` inside the
   CV pipeline, or it leaks.
6. Derived features (for example BMI from `BM1` and `BM2`) are feature
   engineering, a future separate module, not recoding.